# Project: Spatial Prediction of Dengue Clusters in Singapore
### SC3021 Lab Deliverable
**Team:** Jerome, Sathwiik, Marvin
**Stakeholders:** NEA / Public Health Officials

**Research Question:**
Can we predict the emergence and intensity of dengue clusters 2-4 weeks in advance
by analysing localised meteorological patterns?

## 0. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import glob
import os
import sqlite3

import geopandas as gpd
import matplotlib.pyplot as plt
from scipy.spatial import KDTree

# Folder paths (update these if running on Colab)
DENGUE_FOLDERS = [
    "sgcharts/incorrect_latitude_longitude",
    "sgcharts/csv"
]
RAINFALL_2016_FOLDER = "SC3021_rainfall"
RAINFALL_HIST_FILES  = [
    "HistoricalRainfallacrossSingapore2017.csv.download/HistoricalRainfallacrossSingapore2017.csv",
    "HistoricalRainfallacrossSingapore2018.csv.download/HistoricalRainfallacrossSingapore2018.csv",
    "HistoricalRainfallacrossSingapore2019.csv.download/HistoricalRainfallacrossSingapore2019.csv",
]
WEATHER_FILES = [
    "Historical4dayWeatherForecast2016.csv",
    "Historical4dayWeatherForecast2017.csv",
    "Historical4dayWeatherForecast2018.csv",
    "Historical4dayWeatherForecast2019.csv",
]
URA_FILE   = "ura_subzones.geojson"
DATE_START = "2016-03-01"
DATE_END   = "2019-12-31"

---
## DS1 - SGCharts Dengue Cluster Archive
**Source:** sgcharts/csv and sgcharts/incorrect_latitude_longitude

**Schema (no header):** cases_in_location, address, latitude, longitude,
recent_cases_last_2_weeks, cluster_total_cases, cluster_id,
snapshot_yymmdd, active_cluster_count

### 1. Load Dengue Data

In [ ]:
# Read all CSV snapshot files from both folders and combine into one DataFrame
DENGUE_COLS = [
    'cases_in_location', 'address', 'latitude', 'longitude',
    'recent_cases_last_2_weeks', 'cluster_total_cases',
    'cluster_id', 'snapshot_yymmdd', 'active_cluster_count'
]

frames = []
for folder in DENGUE_FOLDERS:
    for f in sorted(glob.glob(os.path.join(folder, "*.csv"))):
        date_part = os.path.basename(f).split('-')[0]
        try:
            snap_date = pd.to_datetime(date_part, format='%y%m%d')
        except ValueError:
            continue
        df = pd.read_csv(f, header=None, encoding='utf-8')
        df = df.dropna(how='all')
        if len(df.columns) != 9:
            continue
        df.columns = DENGUE_COLS
        df['snapshot_date'] = snap_date
        frames.append(df)

dengue_raw = pd.concat(frames, ignore_index=True)
print(f"Loaded {len(dengue_raw)} rows across {dengue_raw['snapshot_date'].nunique()} snapshots")
print(f"Date range: {dengue_raw['snapshot_date'].min().date()} -> {dengue_raw['snapshot_date'].max().date()}")

### 2. Inspect Dengue Data

In [ ]:
print("=== Shape ===")
print(dengue_raw.shape)
print("\n=== Missing Values ===")
print(dengue_raw.isnull().sum())
print("\n=== Snapshots per Year ===")
print(dengue_raw.groupby(dengue_raw['snapshot_date'].dt.year)['snapshot_date'].nunique())
print("\n=== Sample ===")
print(dengue_raw.head(3).to_string())

### 3. Clean Dengue Data

In [ ]:
dengue = dengue_raw.copy()

# Force numeric types on coordinate and case columns
for col in ['latitude', 'longitude', 'cases_in_location',
            'recent_cases_last_2_weeks', 'cluster_total_cases']:
    dengue[col] = pd.to_numeric(dengue[col], errors='coerce')

# Fix the one confirmed bad coordinate:
# upper serangoon crescent block 470a was ~5.2km off in the incorrect_latitude_longitude folder
mask = dengue['address'] == 'upper serangoon crescent (block 470a)'
dengue.loc[mask, 'latitude']  = 1.3792
dengue.loc[mask, 'longitude'] = 103.9017

# Drop placeholder -1 recent_cases (pre-Dec 2013 data artefact)
dengue = dengue[dengue['recent_cases_last_2_weeks'] >= 0]

# Drop rows with missing coordinates
dengue = dengue.dropna(subset=['latitude', 'longitude'])

# Remove duplicates that arise from the two folders overlapping in 2013-2015
dengue = dengue.drop_duplicates(subset=['snapshot_date', 'address', 'cluster_id'])

# Restrict to study window: Mar 2016 - Dec 2019
dengue = dengue[(dengue['snapshot_date'] >= DATE_START) &
                (dengue['snapshot_date'] <= DATE_END)]

# Keep only coordinates within Singapore bounding box
dengue = dengue[dengue['latitude'].between(1.1, 1.5) &
                dengue['longitude'].between(103.6, 104.1)]

# Add ISO year and week — isocalendar year ensures week 1 maps to the correct year
dengue['year']     = dengue['snapshot_date'].dt.isocalendar().year.astype(int)
dengue['iso_week'] = dengue['snapshot_date'].dt.isocalendar().week.astype(int)

# Where multiple snapshots fall in the same ISO week keep only the latest per address
# (each snapshot is point-in-time state so the latest supersedes earlier ones)
dengue = dengue.sort_values('snapshot_date')
dengue = dengue.drop_duplicates(subset=['address', 'year', 'iso_week'], keep='last')

print(f"Clean dengue: {dengue.shape}")
print(f"Date range:   {dengue['snapshot_date'].min().date()} -> {dengue['snapshot_date'].max().date()}")
print(f"Snapshots:    {dengue['snapshot_date'].nunique()}")
print(f"Unique addresses: {dengue['address'].nunique()}")

---
## DS2 - Rainfall Data
**2016 source:** SC3021_rainfall - daily totals, one CSV per station per month

**2017-2019 source:** HistoricalRainfallacrossSingapore - 5-minute readings
aggregated to daily totals on load to keep memory manageable

### 4. Load Rainfall 2016

In [ ]:
# Filenames: DAILYDATA_S{id}_{YYYYMM}.csv
# Extract station_id from filename (e.g. S06) to match 2017-2019 station metadata
rain_2016_frames = []
for f in sorted(glob.glob(os.path.join(RAINFALL_2016_FOLDER, "*.csv"))):
    parts      = os.path.basename(f).replace('.csv', '').split('_')
    station_id = parts[1]

    df = pd.read_csv(f, encoding='latin1')
    df.columns = df.columns.str.strip()

    # Rainfall column name has encoding issues in the header - find it by keyword
    rainfall_col = [c for c in df.columns if 'Rainfall' in c and 'Daily' in c][0]
    df = df.rename(columns={rainfall_col: 'daily_rainfall_mm', 'Station': 'station_name'})

    df['date']              = pd.to_datetime(df[['Year', 'Month', 'Day']])
    df['station_id']        = station_id
    df['daily_rainfall_mm'] = pd.to_numeric(df['daily_rainfall_mm'], errors='coerce').fillna(0)

    rain_2016_frames.append(df[['date', 'station_id', 'station_name', 'daily_rainfall_mm']])

rain_2016 = pd.concat(rain_2016_frames, ignore_index=True)
print(f"2016 rainfall: {len(rain_2016)} rows, {rain_2016['station_id'].nunique()} stations")

### 5. Load Rainfall 2017-2019

In [ ]:
# Each file has ~5M rows of 5-minute readings
# Aggregate to daily totals immediately on load to keep memory low
# Station coordinates are embedded - extract a lookup table here too
rain_hist_frames      = []
station_lookup_frames = []

for f in RAINFALL_HIST_FILES:
    df = pd.read_csv(f, usecols=[
        'date', 'station_id', 'station_name',
        'location_longitude', 'location_latitude', 'reading_value'
    ])
    df['date']          = pd.to_datetime(df['date'])
    df['reading_value'] = pd.to_numeric(df['reading_value'], errors='coerce').fillna(0)

    # Save unique station coordinates for the spatial lookup
    station_lookup_frames.append(
        df[['station_id', 'station_name', 'location_longitude', 'location_latitude']]
        .drop_duplicates('station_id')
    )

    # Sum 5-minute readings into daily totals per station
    daily = df.groupby(['date', 'station_id', 'station_name'])['reading_value'].sum().reset_index()
    daily.rename(columns={'reading_value': 'daily_rainfall_mm'}, inplace=True)
    rain_hist_frames.append(daily)

rain_hist      = pd.concat(rain_hist_frames, ignore_index=True)
station_lookup = (pd.concat(station_lookup_frames)
                  .drop_duplicates('station_id')
                  .reset_index(drop=True))

print(f"2017-2019 rainfall: {len(rain_hist)} rows, {rain_hist['station_id'].nunique()} stations")
print(f"Station coordinate lookup: {len(station_lookup)} stations")

### 6. Combine & Inspect Rainfall

In [ ]:
# Combine 2016 and 2017-2019 into one daily rainfall DataFrame
rain_all = pd.concat(
    [rain_2016, rain_hist[['date', 'station_id', 'station_name', 'daily_rainfall_mm']]],
    ignore_index=True
)

# Filter to study window
rain_all = rain_all[(rain_all['date'] >= DATE_START) & (rain_all['date'] <= DATE_END)]

# Drop station S82 - missing 9 out of 10 months in 2016 making it unreliable
rain_all = rain_all[rain_all['station_id'] != 'S82']

print("=== Daily Rainfall Summary ===")
print(f"Rows:       {len(rain_all)}")
print(f"Stations:   {rain_all['station_id'].nunique()}")
print(f"Date range: {rain_all['date'].min().date()} -> {rain_all['date'].max().date()}")
print("\n=== Missing Values ===")
print(rain_all.isnull().sum())

### 7. Assign Each URA Subzone to its Nearest Rainfall Station
Rainfall is spatially variable so each subzone gets its own rainfall value
based on the nearest physical station using a KDTree nearest-neighbour search.

In [ ]:
# Load URA subzones
ura_gdf = gpd.read_file(URA_FILE)
if ura_gdf.crs is None or ura_gdf.crs.to_epsg() != 4326:
    ura_gdf = ura_gdf.set_crs(epsg=4326)

# Compute subzone centroid coordinates
ura_gdf['centroid'] = ura_gdf.geometry.centroid
subzone_coords = np.array([[p.x, p.y] for p in ura_gdf['centroid']])

# Build station coordinate array from those that appear in rain_all
station_coords_df = station_lookup[
    station_lookup['station_id'].isin(rain_all['station_id'])
].dropna(subset=['location_longitude', 'location_latitude']).reset_index(drop=True)

station_coords = station_coords_df[['location_longitude', 'location_latitude']].values

# Find nearest station for every subzone centroid
tree = KDTree(station_coords)
_, idx = tree.query(subzone_coords)
ura_gdf['nearest_station_id'] = station_coords_df.iloc[idx]['station_id'].values

print(f"Subzones assigned: {len(ura_gdf)}")
print(f"Unique stations used: {ura_gdf['nearest_station_id'].nunique()}")
print(ura_gdf[['SUBZONE_N', 'PLN_AREA_N', 'nearest_station_id']].head(8).to_string(index=False))

### 8. Aggregate Rainfall to Weekly per Subzone

In [ ]:
# Step 1: sum daily rainfall to weekly totals per station
rain_all['year']     = rain_all['date'].dt.isocalendar().year.astype(int)
rain_all['iso_week'] = rain_all['date'].dt.isocalendar().week.astype(int)

rain_weekly_station = (
    rain_all
    .groupby(['year', 'iso_week', 'station_id'])['daily_rainfall_mm']
    .sum()
    .reset_index()
    .rename(columns={'daily_rainfall_mm': 'weekly_rainfall_mm'})
)

# Step 2: map each subzone to its nearest station then pull in weekly rainfall
subzone_station_map = ura_gdf[['SUBZONE_N', 'PLN_AREA_N', 'REGION_N', 'nearest_station_id']].copy()

rain_weekly_subzone = subzone_station_map.merge(
    rain_weekly_station,
    left_on='nearest_station_id',
    right_on='station_id',
    how='left'
).drop(columns='station_id')

print(f"Weekly rainfall per subzone shape: {rain_weekly_subzone.shape}")
print(f"Subzones with rainfall data: {rain_weekly_subzone['SUBZONE_N'].nunique()}")
print(rain_weekly_subzone.head(5).to_string(index=False))

---
## DS3 - Temperature & Humidity (Island-wide)
**Source:** Historical 4-day Weather Forecast 2016-2019

Temperature and humidity are treated as consistent across Singapore -
one value per week island-wide. We retain high, low, average and range
to capture both central tendency and daily variability.

### 9. Load Temperature & Humidity Data

In [ ]:
weather_frames = []
for f in WEATHER_FILES:
    df = pd.read_csv(f)
    df['forecast_date'] = pd.to_datetime(df['forecast_date'])
    weather_frames.append(df[[
        'forecast_date',
        'temperature_high', 'temperature_low',
        'relative_humidity_high', 'relative_humidity_low'
    ]])

weather_raw = pd.concat(weather_frames, ignore_index=True)
weather_raw = weather_raw[(weather_raw['forecast_date'] >= DATE_START) &
                           (weather_raw['forecast_date'] <= DATE_END)]

# Multiple forecast entries may cover the same date - take the mean
weather_daily = weather_raw.groupby('forecast_date').mean().reset_index()

print(f"Daily weather rows: {len(weather_daily)}")
print(f"Date range: {weather_daily['forecast_date'].min().date()} -> {weather_daily['forecast_date'].max().date()}")

### 10. Inspect Weather Data

In [ ]:
print("=== Missing Values ===")
print(weather_daily.isnull().sum())
print("\n=== Distributions ===")
print(weather_daily.describe().round(2))

### 11. Aggregate Weather to Weekly (Island-wide)

In [ ]:
weather_daily['year']     = weather_daily['forecast_date'].dt.isocalendar().year.astype(int)
weather_daily['iso_week'] = weather_daily['forecast_date'].dt.isocalendar().week.astype(int)

weather_weekly = weather_daily.groupby(['year', 'iso_week']).agg(
    temp_high      = ('temperature_high',       'mean'),
    temp_low       = ('temperature_low',        'mean'),
    humidity_high  = ('relative_humidity_high', 'mean'),
    humidity_low   = ('relative_humidity_low',  'mean'),
).reset_index()

# Derive averages and ranges from the weekly highs and lows
weather_weekly['temp_avg']       = (weather_weekly['temp_high'] + weather_weekly['temp_low']) / 2
weather_weekly['temp_range']     = weather_weekly['temp_high'] - weather_weekly['temp_low']
weather_weekly['humidity_avg']   = (weather_weekly['humidity_high'] + weather_weekly['humidity_low']) / 2
weather_weekly['humidity_range'] = weather_weekly['humidity_high'] - weather_weekly['humidity_low']

print(f"Weekly weather shape: {weather_weekly.shape}")
print(weather_weekly.head(5).to_string(index=False))

---
## DS4 - URA Master Plan 2019 Subzone Boundaries
**Source:** ura_subzones.geojson (downloaded from data.gov.sg)

332 subzones across 55 planning areas and 5 regions. Used as the spatial
unit for both dengue aggregation and rainfall station assignment.

### 12. Inspect URA Subzones

In [ ]:
# ura_gdf was already loaded in Section 7
print(f"Subzones:        {len(ura_gdf)}")
print(f"Planning areas:  {ura_gdf['PLN_AREA_N'].nunique()}")
print(f"Regions:         {ura_gdf['REGION_N'].nunique()}")
print(f"CRS:             {ura_gdf.crs}")
print(f"Null geometries: {ura_gdf.geometry.isnull().sum()}")

fig, ax = plt.subplots(figsize=(10, 8))
ura_gdf.plot(ax=ax, color='lightblue', edgecolor='grey', linewidth=0.3, alpha=0.7)
plt.title("Singapore URA Subzones - Master Plan 2019")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.tight_layout()
plt.show()

---
## Data Preparation

### 13. Spatial Join - Dengue Points to URA Subzones

In [ ]:
# Convert dengue lat/lon to GeoDataFrame and assign each point to a subzone
dengue_gdf = gpd.GeoDataFrame(
    dengue,
    geometry=gpd.points_from_xy(dengue['longitude'], dengue['latitude']),
    crs="EPSG:4326"
)

dengue_gdf = gpd.sjoin(
    dengue_gdf,
    ura_gdf[['SUBZONE_N', 'PLN_AREA_N', 'REGION_N', 'geometry']],
    how='left',
    predicate='within'
)

unmatched = dengue_gdf['SUBZONE_N'].isnull().sum()
print(f"Matched:  {len(dengue_gdf) - unmatched} / {len(dengue_gdf)}")
print(f"Dropped:  {unmatched} ({100 * unmatched / len(dengue_gdf):.1f}%)")

dengue_gdf = dengue_gdf.dropna(subset=['SUBZONE_N'])

fig, ax = plt.subplots(figsize=(10, 8))
ura_gdf.plot(ax=ax, color='lightyellow', edgecolor='grey', linewidth=0.3, alpha=0.6)
dengue_gdf.plot(ax=ax, color='red', markersize=2, alpha=0.3)
plt.title("Dengue Cluster Locations (Mar 2016 - Dec 2019)")
plt.tight_layout()
plt.show()

### 14. Aggregate Dengue to Weekly per Subzone

In [ ]:
# One row per subzone per week - sum cases_in_location across all addresses in the subzone
dengue_weekly = (
    dengue_gdf
    .groupby(['year', 'iso_week', 'SUBZONE_N', 'PLN_AREA_N', 'REGION_N'])
    .agg(weekly_cases=('cases_in_location', 'sum'))
    .reset_index()
)

print(f"Weekly dengue shape:        {dengue_weekly.shape}")
print(f"Subzones with cases:        {dengue_weekly['SUBZONE_N'].nunique()}")
print(f"Year-weeks with cases:      {dengue_weekly[['year', 'iso_week']].drop_duplicates().shape[0]}")
print(dengue_weekly.head(5).to_string(index=False))

---
## Feature Engineering
The Aedes mosquito breeding cycle plus dengue viral incubation is approximately
14 days. We shift rainfall and weather features back 2 weeks so conditions at
time T are used to predict dengue cases at time T+2.

### 15. Apply 2-Week Lag to Rainfall (per Subzone)

In [ ]:
# Sort by subzone then time so the shift stays within each subzone independently
rain_weekly_subzone = rain_weekly_subzone.sort_values(['SUBZONE_N', 'year', 'iso_week'])
rain_weekly_subzone['rainfall_lag2'] = (
    rain_weekly_subzone.groupby('SUBZONE_N')['weekly_rainfall_mm'].shift(2)
)

print("Rainfall columns:", ['weekly_rainfall_mm', 'rainfall_lag2'])
print(rain_weekly_subzone[
    ['SUBZONE_N', 'year', 'iso_week', 'weekly_rainfall_mm', 'rainfall_lag2']
].head(8).to_string(index=False))

### 16. Apply 2-Week Lag to Temperature & Humidity (Island-wide)

In [ ]:
weather_weekly = weather_weekly.sort_values(['year', 'iso_week'])

for col in ['temp_avg', 'temp_high', 'temp_low', 'temp_range',
            'humidity_avg', 'humidity_high', 'humidity_low', 'humidity_range']:
    weather_weekly[f'{col}_lag2'] = weather_weekly[col].shift(2)

lag_cols = [c for c in weather_weekly.columns if 'lag2' in c]
print("Lagged weather columns:", lag_cols)
print(weather_weekly[
    ['year', 'iso_week', 'temp_avg', 'temp_avg_lag2', 'humidity_avg', 'humidity_avg_lag2']
].head(5).to_string(index=False))

---
## Merge - Final Analytical Table
**Structure:** one row per subzone x week

Rainfall varies by subzone (nearest-station assignment).
Temperature and humidity are island-wide - the same weekly value is
broadcast across all subzones for that week.

### 17. Merge Dengue + Rainfall + Weather

In [ ]:
# Step 1: dengue weekly LEFT JOIN rainfall weekly on subzone + year + week
merged = dengue_weekly.merge(
    rain_weekly_subzone[['SUBZONE_N', 'year', 'iso_week', 'weekly_rainfall_mm', 'rainfall_lag2']],
    on=['SUBZONE_N', 'year', 'iso_week'],
    how='left'
)

# Step 2: JOIN island-wide weather on year + week
merged = merged.merge(
    weather_weekly,
    on=['year', 'iso_week'],
    how='left'
)

print(f"Final merged shape: {merged.shape}")
print(f"\nColumns:\n{merged.columns.tolist()}")
print(f"\nMissing values:\n{merged.isnull().sum()}")
print(f"\nSample:\n{merged.head(5).to_string(index=False)}")

### 18. Final Table Summary

In [ ]:
print("=== Final Analytical Table ===")
print(f"Total rows:               {len(merged)}")
print(f"Subzones covered:         {merged['SUBZONE_N'].nunique()}")
print(f"Year-weeks covered:       {merged[['year', 'iso_week']].drop_duplicates().shape[0]}")
print(f"Years:                    {sorted(merged['year'].unique())}")
print(f"Total weekly cases:       {merged['weekly_cases'].sum():.0f}")
print(f"Avg cases/subzone-week:   {merged['weekly_cases'].mean():.2f}")

print("\n=== Descriptive Statistics ===")
print(merged[[
    'weekly_cases', 'weekly_rainfall_mm', 'rainfall_lag2',
    'temp_avg', 'temp_range', 'humidity_avg', 'humidity_range'
]].describe().round(2))

---
## Visualisation

### 19. Dengue Cases vs Lagged Rainfall, Temperature and Humidity

In [ ]:
# Aggregate to island-wide weekly totals for time-series plots
island_weekly = merged.groupby(['year', 'iso_week']).agg(
    total_cases       = ('weekly_cases',   'sum'),
    avg_rainfall_lag2 = ('rainfall_lag2',  'mean'),
    temp_avg          = ('temp_avg',       'first'),
    temp_range        = ('temp_range',     'first'),
    humidity_avg      = ('humidity_avg',   'first'),
    humidity_range    = ('humidity_range', 'first'),
).reset_index()

island_weekly['label'] = (island_weekly['year'].astype(str) + '-W' +
                          island_weekly['iso_week'].astype(str).str.zfill(2))
x = range(len(island_weekly))

fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Plot 1: Cases vs 2-week lagged rainfall
ax1, ax2 = axes[0], axes[0].twinx()
ax1.plot(x, island_weekly['total_cases'],       color='red',  label='Weekly Cases')
ax2.plot(x, island_weekly['avg_rainfall_lag2'], color='blue', label='Rainfall Lag-2 (mm)', alpha=0.6)
ax1.set_ylabel('Dengue Cases', color='red')
ax2.set_ylabel('Rainfall mm (2-week lag)', color='blue')
axes[0].set_title('Weekly Dengue Cases vs 2-Week Lagged Rainfall')
axes[0].set_xticks(list(x)[::8])
axes[0].set_xticklabels(island_weekly['label'].iloc[::8], rotation=45, ha='right', fontsize=7)

# Plot 2: Cases vs temperature average and range
ax3, ax4 = axes[1], axes[1].twinx()
ax3.plot(x, island_weekly['total_cases'], color='red',    label='Weekly Cases')
ax4.plot(x, island_weekly['temp_avg'],    color='orange', label='Temp Avg', alpha=0.7)
ax4.fill_between(x,
    island_weekly['temp_avg'] - island_weekly['temp_range'] / 2,
    island_weekly['temp_avg'] + island_weekly['temp_range'] / 2,
    alpha=0.15, color='orange', label='Temp Range')
ax3.set_ylabel('Dengue Cases', color='red')
ax4.set_ylabel('Temperature C', color='orange')
axes[1].set_title('Weekly Dengue Cases vs Temperature (Avg & Range)')
axes[1].set_xticks(list(x)[::8])
axes[1].set_xticklabels(island_weekly['label'].iloc[::8], rotation=45, ha='right', fontsize=7)

# Plot 3: Cases vs humidity average and range
ax5, ax6 = axes[2], axes[2].twinx()
ax5.plot(x, island_weekly['total_cases'],  color='red',   label='Weekly Cases')
ax6.plot(x, island_weekly['humidity_avg'], color='green', label='Humidity Avg', alpha=0.7)
ax6.fill_between(x,
    island_weekly['humidity_avg'] - island_weekly['humidity_range'] / 2,
    island_weekly['humidity_avg'] + island_weekly['humidity_range'] / 2,
    alpha=0.15, color='green', label='Humidity Range')
ax5.set_ylabel('Dengue Cases', color='red')
ax6.set_ylabel('Relative Humidity %', color='green')
axes[2].set_title('Weekly Dengue Cases vs Humidity (Avg & Range)')
axes[2].set_xticks(list(x)[::8])
axes[2].set_xticklabels(island_weekly['label'].iloc[::8], rotation=45, ha='right', fontsize=7)

plt.tight_layout()
plt.savefig('dengue_weather_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Data Storage

### 20. Store Final Table in SQLite

In [ ]:
if os.path.exists('dengue_weather_sg.db'):
    os.remove('dengue_weather_sg.db')

conn = sqlite3.connect('dengue_weather_sg.db')

merged.to_sql('dengue_weather_weekly', conn, if_exists='replace', index=False)
ura_gdf.drop(columns=['geometry', 'centroid']).to_sql('ura_planning_areas', conn, if_exists='replace', index=False)

tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print("Tables saved:", tables['name'].tolist())
for t in tables['name']:
    n = pd.read_sql(f"SELECT COUNT(*) as n FROM {t}", conn).iloc[0]['n']
    print(f"  {t}: {n} rows")

conn.close()
print("\nSaved to dengue_weather_sg.db")